In [1]:
!pip install -q langchain langchain-community langchainhub langchain-chroma beautifulsoup4 langchain_google_genai langchain-classic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 k

In [2]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
LANGSMITH_API_KEY = userdata.get('LANGSMITH_API_KEY')

import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_PROJECT"] = "RAG_w_conversation"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
gemini_embed = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model = "gemini-3.1-flash-lite")

In [34]:
import bs4
from langchain_classic import hub
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import MessagesPlaceholder

In [35]:
import langchain
print(langchain.__version__)

1.3.6


In [36]:
loader = WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
                       bs_kwargs = dict(parse_only=bs4.SoupStrainer(class_=("post-content","post-title","post-header"))))

In [37]:
doc = loader.load()

In [38]:
text_splitte = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap =200)
splits = text_splitte.split_documents(doc)

In [39]:
vectorstore = Chroma.from_documents(documents=splits, embedding=gemini_embed)
retriever = vectorstore.as_retriever()

In [40]:
system_prompt = (
    "you are an assistant for question-answering tasks. "
    "Use the following pieces of context to answer the question at the end. "
    "If you don't know the answer, just say that you don't know, don't try to make up an answer. "
    "\n\n"
    "{context}"
)

In [41]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [42]:
question_answering_chain = create_stuff_documents_chain(model, chat_prompt)

In [43]:
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [44]:
response = rag_chain.invoke({"input":"What is MKRL?"})

In [45]:
response["answer"]

'MRKL (Karpas et al. 2022) stands for "Modular Reasoning, Knowledge and Language." It is a neuro-symbolic architecture designed for autonomous agents.\n\nIn a MRKL system, a general-purpose Large Language Model (LLM) acts as a router that directs inquiries to a collection of "expert" modules. These expert modules can be neural (such as deep learning models) or symbolic (such as a math calculator, currency converter, or weather API).'

In [46]:
from langchain_classic.chains import create_history_aware_retriever

In [47]:
retriever_prompt = (
    "Given a chat history and the latest user question which might reference "
    "context in the chat history, formulate a standalone question that can be "
    "understood without the chat history. Do NOT answer the question. "
    "If the question is already standalone, return it exactly as is. "
    "You MUST always return a question - never return an empty response."
)

In [48]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", retriever_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [62]:
from langchain_core.runnables import RunnableLambda

def route_query(x):
    # No history → use input directly
    if not x.get("chat_history"):
        return x["input"]
    # Has history → reformulate the question
    messages = contextualize_q_prompt.format_messages(
        chat_history=x["chat_history"],
        input=x["input"]
    )
    result = model.invoke(messages)

    # Handle content being a list (Gemini thinking models return [thinking_part, text_part])
    if isinstance(result.content, list):
        query = "".join(
            part["text"] for part in result.content
            if isinstance(part, dict) and part.get("type") == "text"
        )
    else:
        query = result.content

    # Fallback if model returns empty
    if not query or not query.strip():
        return x["input"]
    return query

history_aware_retriever = RunnableLambda(route_query) | retriever

In [63]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt =  ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [64]:
qa_chain = create_stuff_documents_chain(model, qa_prompt)

In [65]:
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

In [66]:
from langchain_core.messages import HumanMessage, AIMessage

In [67]:
chat_history = []

In [68]:
q1 = "what is task decomposition?"

In [69]:
m1 = rag_chain.invoke({"input":q1, "chat_history":chat_history})

In [70]:
m1["answer"]

'Task decomposition is a planning technique used to break down a complicated task—which typically involves many steps—into smaller, more manageable subgoals or steps.\n\nAccording to the provided text, task decomposition can be accomplished through the following methods:\n\n1.  **LLM Prompting:** Using simple prompts such as "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n2.  **Task-specific instructions:** Providing tailored guidance, such as "Write a story outline" for the purpose of writing a novel.\n3.  **Human inputs:** Integrating guidance directly from human users.\n4.  **Chain of Thought (CoT):** A prompting technique where the model is instructed to "think step by step" to decompose complex tasks into simpler steps, allowing for greater test-time computation.\n5.  **Tree of Thoughts (ToT):** An extension of CoT that explores multiple reasoning possibilities at each step by generating multiple thoughts per step to create a tree structure, which is then navi

In [71]:
chat_history.append(HumanMessage(content=q1))
chat_history.append(AIMessage(content=m1["answer"]))

In [72]:
chat_history

[HumanMessage(content='what is task decomposition?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Task decomposition is a planning technique used to break down a complicated task—which typically involves many steps—into smaller, more manageable subgoals or steps.\n\nAccording to the provided text, task decomposition can be accomplished through the following methods:\n\n1.  **LLM Prompting:** Using simple prompts such as "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n2.  **Task-specific instructions:** Providing tailored guidance, such as "Write a story outline" for the purpose of writing a novel.\n3.  **Human inputs:** Integrating guidance directly from human users.\n4.  **Chain of Thought (CoT):** A prompting technique where the model is instructed to "think step by step" to decompose complex tasks into simpler steps, allowing for greater test-time computation.\n5.  **Tree of Thoughts (ToT):** An extension of CoT that explores multiple reasoni

In [73]:
q2 = "what are common ways of doing it?"
m2 = rag_chain.invoke({"input":q2, "chat_history":chat_history})
print(m2)

{'input': 'what are common ways of doing it?', 'chat_history': [HumanMessage(content='what is task decomposition?', additional_kwargs={}, response_metadata={}), AIMessage(content='Task decomposition is a planning technique used to break down a complicated task—which typically involves many steps—into smaller, more manageable subgoals or steps.\n\nAccording to the provided text, task decomposition can be accomplished through the following methods:\n\n1.  **LLM Prompting:** Using simple prompts such as "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n2.  **Task-specific instructions:** Providing tailored guidance, such as "Write a story outline" for the purpose of writing a novel.\n3.  **Human inputs:** Integrating guidance directly from human users.\n4.  **Chain of Thought (CoT):** A prompting technique where the model is instructed to "think step by step" to decompose complex tasks into simpler steps, allowing for greater test-time computation.\n5.  **Tree of Though

In [74]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [75]:
store ={}

In [76]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
  if session_id not in store:
    store[session_id] = ChatMessageHistory()
  return store[session_id]

In [77]:
conv_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    output_messages_key="answer",
    history_messages_key="chat_history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [78]:
conv_rag_chain.invoke({"input":"what is task decomposition?"},
                      config={"configurable":{"session_id":"1"}},)["answer"]

'Task decomposition is a process used to break down a complicated task into smaller, more manageable steps or subgoals. \n\nBased on the provided text, it can be achieved through several methods:\n\n*   **LLM prompting:** Using simple prompts such as "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n*   **Task-specific instructions:** Providing instructions tailored to a specific task, such as "Write a story outline" for the purpose of writing a novel.\n*   **Human input:** Incorporating direct guidance from humans.\n*   **Chain of Thought (CoT):** Instructing the model to "think step by step," which allows it to use test-time computation to turn complex tasks into simpler components.\n*   **Tree of Thoughts (ToT):** Extending CoT by exploring multiple reasoning possibilities at each step, generating multiple thoughts per step to create a tree structure that can be navigated via search algorithms like BFS or DFS.'

In [79]:
store

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='what is task decomposition?', additional_kwargs={}, response_metadata={}), AIMessage(content='Task decomposition is a process used to break down a complicated task into smaller, more manageable steps or subgoals. \n\nBased on the provided text, it can be achieved through several methods:\n\n*   **LLM prompting:** Using simple prompts such as "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n*   **Task-specific instructions:** Providing instructions tailored to a specific task, such as "Write a story outline" for the purpose of writing a novel.\n*   **Human input:** Incorporating direct guidance from humans.\n*   **Chain of Thought (CoT):** Instructing the model to "think step by step," which allows it to use test-time computation to turn complex tasks into simpler components.\n*   **Tree of Thoughts (ToT):** Extending CoT by exploring multiple reasoning possibilities at each step, generating multiple th

In [80]:
conv_rag_chain.invoke({"input":"what are common ways of doing it?"},
                      config={"configurable":{"session_id":"1"}},)["answer"]

'According to the provided text, the common ways to perform task decomposition include:\n\n*   **Simple LLM Prompting:** Using direct prompts such as "Steps for XYZ.\\n1." or asking the model to identify the subgoals needed to achieve a task.\n*   **Task-Specific Instructions:** Providing specialized prompts for specific domains, such as "Write a story outline" when the goal is to write a novel.\n*   **Human Inputs:** Incorporating direct guidance from humans to break down tasks.\n*   **Chain of Thought (CoT):** Using the technique of instructing the model to "think step by step." This leverages test-time computation to decompose large, complex tasks into smaller, manageable sub-steps.\n*   **Tree of Thoughts (ToT):** An extension of CoT that explores multiple reasoning possibilities at each step. By generating multiple thoughts per step, it creates a tree structure that can be navigated using search algorithms like breadth-first search (BFS) or depth-first search (DFS).'

In [81]:
store

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='what is task decomposition?', additional_kwargs={}, response_metadata={}), AIMessage(content='Task decomposition is a process used to break down a complicated task into smaller, more manageable steps or subgoals. \n\nBased on the provided text, it can be achieved through several methods:\n\n*   **LLM prompting:** Using simple prompts such as "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?"\n*   **Task-specific instructions:** Providing instructions tailored to a specific task, such as "Write a story outline" for the purpose of writing a novel.\n*   **Human input:** Incorporating direct guidance from humans.\n*   **Chain of Thought (CoT):** Instructing the model to "think step by step," which allows it to use test-time computation to turn complex tasks into simpler components.\n*   **Tree of Thoughts (ToT):** Extending CoT by exploring multiple reasoning possibilities at each step, generating multiple th